# 04_make_fingerprints — fingerprint 4종 계산

**한 줄 요약:** `same_dedup_keepdiff` 시트의 분자들에 대해 **4가지 fingerprint**(ECFP4·MACCS·RDKit·AtomPair)를 계산해 각각 시트로 저장한다.
**용어:** fingerprint=분자 구조를 0/1 비트 배열로 나타낸 지문. 종류마다 구조를 다르게 요약.
**큰 흐름:** ① 준비·읽기 → ② 유효 분자만 → ③ 생성기 준비 → ④ 4종 지문 계산·저장

> **📌 이 노트북 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]** 순서.
> ③은 그 셀에 **처음 나온** 함수·문법 설명(기초 반복은 *(01에서 설명)* 으로 생략). `# ...`=주석.

### 준비 — 폴더 위치 맞추기
노트북을 어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동한다(그래야 `data/...` 경로가 맞음).

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 **코드 뜯어보기** *(01에서 설명)*: `os.chdir('..')`=상위 폴더로 이동, `os.path.isdir`=폴더 존재 확인, `print`=화면 출력.

### 셀 1 — 준비 + 데이터 읽기
라이브러리를 불러오고 경로·비트수를 정한 뒤, 3번째 시트를 표로 읽는다.

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

SRC = "data/HSD17B13_IC50_merged.xlsx"
OUT = "data/HSD17B13_fingerprints.xlsx"
NBITS = 1024

df = pd.read_excel(SRC, sheet_name="same_dedup_keepdiff")

🔎 **코드 뜯어보기 (셀 1)** *(import/read_excel는 01에서 설명)*
- `from rdkit import Chem, DataStructs` : 한 줄에 **여러 개**를 가져오기. `DataStructs`=지문 자료형 변환용.
- `from rdkit.Chem import MACCSkeys, rdFingerprintGenerator` : MACCS 지문·지문 생성기 도구. `NBITS = 1024`=지문 길이.

### 셀 2 — 기준 열 지정 + 유효 분자만
결과에 함께 넣을 정보 열(meta)을 정하고, SMILES가 있는 행만 남긴다.

In [ ]:
meta_cols = ["canonical_smiles", "ic50_nM", "relation", "sources"]
df = df.dropna(subset=["canonical_smiles"]).reset_index(drop=True)

🔎 **코드 뜯어보기 (셀 2)**
- `meta_cols = [...]` : 결과에 붙일 정보 열 이름 목록. `.dropna(subset=["canonical_smiles"]).reset_index(drop=True)`=SMILES 없는 행 제거 후 번호 새로.

### 셀 3 — fingerprint 생성기 3개 + MACCS 변환 함수
지문을 만드는 생성기를 미리 만들고(재사용), MACCS 지문을 배열로 바꾸는 함수를 정의한다.

In [ ]:
gen_ecfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)
gen_rdk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=NBITS)
gen_ap = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=NBITS)


def maccs_np(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

🔎 **코드 뜯어보기 (셀 3)**
- `rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)` : **ECFP4** 생성기(원자 주변 반경 2). `GetRDKitFPGenerator`=경로형, `GetAtomPairGenerator`=원자쌍형 생성기.
- `def maccs_np(mol):` : MACCS 지문을 numpy 배열로 바꾸는 함수. `MACCSkeys.GenMACCSKeys(mol)`=MACCS 지문 계산, `np.zeros((n,), dtype=np.int8)`=0으로 채운 배열, `DataStructs.ConvertToNumpyArray(fp, arr)`=지문을 배열 arr에 채워 넣기.

### 셀 4 — 분자마다 4종 지문 계산 → 시트별로 저장
분자를 한 번씩만 읽어 4가지 지문을 동시에 계산하고, 종류별 시트로 엑셀에 저장한다.

In [ ]:
rows = {"ECFP4": [], "MACCS": [], "RDKit": [], "AtomPair": []}
keep_idx = []
for i, smi in enumerate(df["canonical_smiles"]):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    keep_idx.append(i)
    rows["ECFP4"].append(gen_ecfp.GetFingerprintAsNumPy(mol))
    rows["RDKit"].append(gen_rdk.GetFingerprintAsNumPy(mol))
    rows["AtomPair"].append(gen_ap.GetFingerprintAsNumPy(mol))
    rows["MACCS"].append(maccs_np(mol))

meta = df.loc[keep_idx, meta_cols].reset_index(drop=True)

with pd.ExcelWriter(OUT, engine="openpyxl") as w:
    for name, mat in rows.items():
        X = np.vstack(mat)
        cols = [f"X{j+1}" for j in range(X.shape[1])]
        fp_df = pd.concat([meta, pd.DataFrame(X, columns=cols)], axis=1)
        fp_df.to_excel(w, sheet_name=name, index=False)
        print(f"[{name}] {X.shape[0]}행 x {X.shape[1]}bit")

print("\n저장 완료 →", OUT)
print(f"입력 {len(df)}개 중 {len(keep_idx)}개 정상 변환")

🔎 **코드 뜯어보기 (셀 4)**
- `rows = {"ECFP4": [], ...}` : **딕셔너리에 빈 리스트 4개** — 지문 종류별로 값을 모을 곳.
- `for i, smi in enumerate(df["canonical_smiles"]):` : 분자를 하나씩 반복(01에서 설명).
- `gen_ecfp.GetFingerprintAsNumPy(mol)` : 생성기로 지문을 0/1 배열로 계산해 해당 리스트에 추가.
- `np.vstack(mat)` : 배열들을 위아래로 쌓아 표로. `[f"X{j+1}" for j in range(...)]`=열 이름 X1,X2,…
- `for name, mat in rows.items():` : 지문 종류(name)와 값(mat)을 함께 반복하며 각각 **시트로 저장**(01의 ExcelWriter 참고).